# S2 - In-Hand Cube Reorientation (LEAP hand, MJX)

Trains a PPO policy to reorient a 5 cm cube held in a LEAP right hand.

**Before you run anything:**

1. Settings -> Accelerator -> **GPU T4 x2** (or P100).
2. Settings -> Internet -> **On** (needed for pip and the git clone).
3. Free tier gives ~30 GPU-hours/week and kills the session at ~12 hours.

**This notebook is designed to be run many times.** Each session resumes
from the last checkpoint. See the final cell for how to chain them.

---

### Read this before burning GPU hours

The training loop has **never completed a single iteration anywhere**. It
was written and unit-tested on a CPU box where one `mjx.step` costs ~4 s
and the compile alone runs past 30 minutes, so the loop could not be closed
locally.

Cell 4 is a deliberate gate: a tiny run that must print an iteration line
and write a checkpoint before you start the real thing. Do not skip it. If
it fails you have lost two minutes instead of an afternoon.


## 1. Environment setup


In [ ]:
import glob, os, shutil, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
SRC  = WORK / 'robotics-rl-portfolio'
CKPT = WORK / 'checkpoints'
CKPT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=True)

sh(sys.executable + ' -m pip install -q --upgrade "jax[cuda12]" mujoco mujoco-mjx optax')


In [ ]:
import jax
print('jax', jax.__version__)
print('devices:', jax.devices())

# Hard stop rather than a warning. On CPU this model needs ~4 s per
# mjx.step unbatched, so a CPU session is not a slow run, it is no run.
assert any(d.platform == 'gpu' for d in jax.devices()), \
    'No GPU visible to JAX. Settings -> Accelerator -> GPU, then restart.'
print()
print('GPU OK')


## 2. Get the code

Clones the public portfolio repo. If `s2_inhand/` is missing, the S2 commit
has not been pushed yet. Push it from your machine first:

```
cd ~/Downloads/hermestes/robotics-rl-portfolio && git push origin main
```


In [ ]:
REPO = 'https://github.com/JacobEGarcia/robotics-rl-portfolio.git'

if SRC.exists():
    sh('cd ' + str(SRC) + ' && git pull --ff-only')
else:
    sh('git clone --depth 1 ' + REPO + ' ' + str(SRC))

if not (SRC / 's2_inhand' / 'train.py').exists():
    raise SystemExit(
        's2_inhand/ is not in the cloned repo. Push the S2 commit from '
        'your local machine (git push origin main), then re-run this cell.')

os.environ['PYTHONPATH'] = str(SRC)
print()
print('code OK:', sorted(p.name for p in (SRC / 's2_inhand').iterdir()))


## 2b. Fetch the LEAP hand model

`assets/menagerie/` is gitignored in the portfolio repo on purpose: MuJoCo
Menagerie is 2.3 GB of third-party assets and is itself a git repository,
so it is fetched rather than vendored. **The clone above therefore does not
include the hand model**, and `scene.xml` cannot load without it.

`scripts/fetch_assets.sh` pulls all 2.3 GB. S2 only needs `leap_hand/`, so
this uses a sparse checkout instead and grabs a few MB.


In [ ]:
MENAGERIE = SRC / 'assets' / 'menagerie'

if (MENAGERIE / 'leap_hand' / 'right_hand.xml').exists():
    print('LEAP hand already present')
else:
    MENAGERIE.parent.mkdir(parents=True, exist_ok=True)
    try:
        sh('git clone --depth 1 --filter=blob:none --sparse '
           'https://github.com/google-deepmind/mujoco_menagerie.git '
           + str(MENAGERIE))
        sh('cd ' + str(MENAGERIE) + ' && git sparse-checkout set leap_hand')
    except subprocess.CalledProcessError:
        # Fall back to the repo's own fetch script, which clones the full
        # 2.3 GB. Slower, but it works if sparse checkout is unavailable.
        print('sparse checkout failed, falling back to full clone')
        shutil.rmtree(MENAGERIE, ignore_errors=True)
        sh('cd ' + str(SRC) + ' && bash scripts/fetch_assets.sh')

assert (MENAGERIE / 'leap_hand' / 'right_hand.xml').exists(), \
    'LEAP hand model still missing after fetch'
print('hand model OK')


### Load the scene

Loads `scene.xml` and re-checks the reset grasp. This catches a broken or
partial asset fetch here, in seconds, rather than 20 minutes into a compile.


In [ ]:
import mujoco, numpy as np

m = mujoco.MjModel.from_xml_path(str(SRC / 's2_inhand' / 'scene.xml'))
d = mujoco.MjData(m)
mujoco.mj_resetDataKeyframe(m, d, 0)
mujoco.mj_forward(m, d)

cb = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, 'cube')
cg = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, 'cube_geom')
pen = min((d.contact[c].dist for c in range(d.ncon)
           if cg in (d.contact[c].geom1, d.contact[c].geom2)), default=0.0)
start = d.xpos[cb].copy()
for _ in range(2500):
    mujoco.mj_step(m, d)
drift = float(np.linalg.norm(d.xpos[cb] - start))

print('nq=%d nv=%d nu=%d' % (m.nq, m.nv, m.nu))
print('reset penetration %.5f m   drift over 5 s %.4f m' % (pen, drift))
assert drift < 0.03, 'grasp is not holding; do not train against this'
print('scene OK')


## 3. Resume from a previous session (optional)

To continue a run: **+ Add Input -> Your Work -> Notebook Output** of the
previous version, then re-run. Checkpoints are copied in below.

On the very first session this prints `starting fresh`, which is correct.


In [ ]:
seeded = 0
for pat in ('/kaggle/input/*/checkpoints/ckpt_*.pkl',
            '/kaggle/input/*/ckpt_*.pkl'):
    for f in glob.glob(pat):
        shutil.copy2(f, CKPT / Path(f).name)
        seeded += 1

if seeded:
    latest = sorted(CKPT.glob('ckpt_*.pkl'))[-1]
    print('seeded', seeded, 'checkpoint(s); newest =', latest.name)
else:
    print('no previous session attached, starting fresh')


## 4. Smoke gate - do not skip

Tiny run: 4 envs, a couple of iterations, one checkpoint write. It proves
the rollout, the GAE, the PPO update, and the checkpoint path all work end
to end on this machine.

**It must print at least one `step ...` line and one `checkpoint @ ...`
line.** If it does not, stop and fix that before the real run.


In [ ]:
import time

t0 = time.time()
r = subprocess.run(
    [sys.executable, '-u', '-m', 's2_inhand.train', '--smoke',
     '--ckpt-dir', str(WORK / 'smoke_ckpt')],
    cwd=str(SRC), env=dict(os.environ, PYTHONPATH=str(SRC)),
    capture_output=True, text=True)

print(r.stdout[-4000:])
if r.stderr.strip():
    print('--- stderr ---')
    print(r.stderr[-3000:])
print()
print('smoke exit=%d in %.0fs' % (r.returncode, time.time() - t0))

wrote = list((WORK / 'smoke_ckpt').glob('ckpt_*.pkl'))
assert r.returncode == 0, 'smoke run failed, see stderr above'
assert wrote, 'smoke run wrote no checkpoint'
print('SMOKE GATE PASSED ->', [p.name for p in wrote])


## 5. The real run

`--resume` picks up the newest checkpoint automatically, so this is the
same cell every session.

**Watch the `curr` column, not the reward.** If the curriculum is not
widening after a few million steps, the reward is not shaping behaviour and
the fix is the reward, not more steps. That is the expected failure mode
for this project.

Leave headroom: interrupt this and run cell 6 before the 12-hour cap.


In [ ]:
TOTAL_STEPS = 100_000_000
NUM_ENVS    = 2048     # drop to 1024 if you hit OOM on a T4

cmd = [sys.executable, '-u', '-m', 's2_inhand.train',
       '--total-steps', str(TOTAL_STEPS),
       '--num-envs', str(NUM_ENVS),
       '--ckpt-dir', str(CKPT),
       '--ckpt-every', '2000000',
       '--resume']
print('$', ' '.join(cmd), flush=True)

# Stream output live. A ten-hour run that prints nothing until it exits
# is impossible to babysit, and the whole point is catching a flat
# curriculum in hour one rather than hour eleven.
p = subprocess.Popen(cmd, cwd=str(SRC),
                     env=dict(os.environ, PYTHONPATH=str(SRC)),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    p.terminate()
    print()
    print('interrupted; newest checkpoint is still on disk')
p.wait()


## 6. State of the run, and how to continue


In [ ]:
cks = sorted(CKPT.glob('ckpt_*.pkl'))
print(len(cks), 'checkpoint(s) in', CKPT)
for c in cks[-5:]:
    print('  ', c.name, '%.1f MB' % (c.stat().st_size / 1e6))

meta = CKPT / 'latest.json'
if meta.exists():
    print()
    print(meta.read_text())

print()
print('To continue in a new session:')
print('  1. Save Version (Quick Save is enough after this cell)')
print('  2. New session -> + Add Input -> Your Work -> this notebook output')
print('  3. Run cells 1-3, skip the smoke gate, run cell 5')
